# Notebook 4 — Vector Store + RAG

Semantic search over investment notes and health reference.

**Exit criteria:** Top result is clearly relevant for all 10 test queries.

**Install:** `pip install sentence-transformers numpy`

## 1. Setup

In [ ]:

import sqlite3
import numpy as np
import json
from datetime import datetime

try:
    from sentence_transformers import SentenceTransformer
    MODEL_AVAILABLE = True
    print("sentence-transformers available")
except ImportError:
    MODEL_AVAILABLE = False
    print("sentence-transformers not installed.")
    print("Run: pip install sentence-transformers")

DB_PATH = "second_brain.db"
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()
print(f"Connected to {DB_PATH}")


## 2. Load Embedding Model

In [ ]:

if MODEL_AVAILABLE:
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    print("Model loaded: paraphrase-multilingual-MiniLM-L12-v2")
    
    # Test embedding
    test_emb = model.encode("test sentence")
    print(f"Embedding dimension: {len(test_emb)}")
else:
    print("Using random embeddings for structure test (not meaningful for search)")
    class MockModel:
        def encode(self, text):
            np.random.seed(hash(text) % 2**31)
            return np.random.rand(384).astype(np.float32)
    model = MockModel()


## 3. Content Chunks — Investment Notes

In [ ]:

investment_chunks = [
    {
        'content': "Anand Srinivasan advice on Cipla: Cipla is highly ethical company. Buy below PE 23-24. One of its US plants closed for 6 months for investigation. Cipla is borderline expensive but can consider.",
        'source': 'anand',
        'domain': 'investment',
    },
    {
        'content': "Anand Srinivasan advice on South Indian Bank: New head has track record to turn around things. Strongly suggests buying South Indian Bank.",
        'source': 'anand',
        'domain': 'investment',
    },
    {
        'content': "Anand Srinivasan advice on IndusInd Bank: IndusInd Bank maybe a multi-bagger. Strongly suggests buying IndusInd Bank.",
        'source': 'anand',
        'domain': 'investment',
    },
    {
        'content': "Anand Srinivasan advice on Dr Reddy: Dr Reddy below 20 PE can consider. Strongly suggests buying Dr Reddy.",
        'source': 'anand',
        'domain': 'investment',
    },
    {
        'content': "Anand Srinivasan advice on IDFC First Bank: CEO accepted micro finance as mistake, shows integrity. Strongly suggests buying IDFC First Bank. Micro finance means loan less than 3 lakh, people not paying it, bank can sort out soon.",
        'source': 'anand',
        'domain': 'investment',
    },
    {
        'content': "Anand Srinivasan general advice: Hold ITC, buy at dips. Gold price will increase till 5k dollars. Buy export-based companies like bajaj auto, pharma. IT, pharma and auto doing very good business. Dont buy SBI and LIC stocks.",
        'source': 'anand',
        'domain': 'investment',
    },
    {
        'content': "PR Sundar prediction: Nifty 50 will not improve for at least 3-4 years. If someone invested in nifty 50 last year, would be at loss of 3-4%. Next year needs 16-17% profit to match FD returns of 7%.",
        'source': 'pr sundar',
        'domain': 'investment',
    },
    {
        'content': "Investment mistakes made: Bought Chinese ETF at high price — everyone read same news over weekend and jumped in. Bought ICICI AMC after seeing 87% in one quarter, same mistake. Had waited 5 days would have got it 100rs cheaper. Bought Heritage Foods after Chandrababu Naidu election win like everyone else.",
        'source': 'self',
        'domain': 'investment',
    },
    {
        'content': "Investment strategy that worked: Split desired amount into 3 parts, invest 1/3 initially — need dry powder. Lost conviction on AMD, Inoq, Chinese ETF and paid the price. Should not sell with conviction stocks.",
        'source': 'self',
        'domain': 'investment',
    },
    {
        'content': "48-hour rule for investments: Build deliberate friction into decisions. Add 48-hour rule for any decision involving more than 50k rupees. This is not doubt — it is letting subconscious process what conscious mind missed.",
        'source': 'self',
        'domain': 'investment',
    },
    {
        'content': "Peter Lynch principles: PE between 3 and 6 can hardly fail. Focus on company not on stocks. Without a notebook with reasons for buying a stock, easy to forget why. Rule of 72: divide annual return percentage by 72, result is years to double. When coming out of recession, sell bank and insurance stocks and invest in retailers and auto sectors.",
        'source': 'peter lynch',
        'domain': 'investment',
    },
    {
        'content': "Pharma allocation plan: 40% Natco (PE 8-9x + GLP-1 exclusivity), 25% Dr Reddy (PE 15x + diversification), 20% Divi's Labs (CDMO), 15% Sun Pharma. Avoid Abbott and Lupin (overvalued). Natco is undervalued pharma stock.",
        'source': 'self',
        'domain': 'investment',
    },
    {
        'content': "Current stock holdings Feb 2026: Gold (hedging), Chinese Tech ETF (chinese exposure), South Indian Bank (below book value), Tamilnad Mercantile (below book value), IndusInd Bank (distress turnaround), IDFC First Bank, ICICI AMC, Bajaj Finance (major NBFC), Hero Moto Corp (40% stake in Ather), Waaree Energies, Suzlon, Bharti Airtel, ITC, Natco, Dr Reddy, Sun Pharma, Cipla.",
        'source': 'self',
        'domain': 'investment',
    },
    {
        'content': "Banking metrics reference: NIM above 3% is healthy for banks, above 5% for NBFCs. CAR minimum 9% (RBI Basel III), healthy above 12-13%. Gross NPA below 3% for banks, below 2% is ideal. Net NPA below 1% is healthy. ICR above 3 is healthy, below 1 is bad.",
        'source': 'self',
        'domain': 'investment',
    },
    {
        'content': "Conviction tracking: Guessed Chinese ETF would bounce back — became right. Invested in Tamilnad Mercantile bank when Anand mentioned it subtly — in reasonable profit. Sold AMD before it became big — lost conviction. Sold Inoq before it became big. Sold Chinese ETF at loss, now above investment price.",
        'source': 'self',
        'domain': 'investment',
    },
]

print(f"Investment chunks prepared: {len(investment_chunks)}")


## 4. Content Chunks — Health Reference

In [ ]:

health_chunks = [
    {
        'content': "Root cause of psoriasis: Damaged gut lining leads to leaky gut from alkaloids, gluten, or stress. This causes immune overreaction and excess histamine, resulting in psoriasis, itching, cold symptoms, and acidity. Fix the gut to fix everything else.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "What damages the gut: Antibiotics and NSAIDs like Crocin and Combiflam. Chronic stress. Refined sugar and packaged food. Chlorinated tap water. Late nights — gut repairs during sleep. Milk tea on empty stomach daily.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Foods to avoid strictly — nightshades cause leaky gut through alkaloids: Tomato, brinjal, capsicum, potato. All nightshades damage gut lining and worsen psoriasis.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Foods to avoid — dairy triggers immune response through casein: Milk, paneer, cheese, ice cream to avoid. Butter reduce. Ghee is the exception and is allowed.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Foods to avoid — gluten causes gut holes through zonulin: Wheat roti, chapati, maida, puri, parota, bread, rava, sooji. All gluten-containing foods to avoid.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Foods to avoid — high histamine foods: Dry fish (karuvaadu), shellfish (prawns, crab), pineapple, grapes, strawberry, spinach, pickles, vinegar, aged cheese, dry pattani, sweet corn, peanuts. Also avoid refined sugar, refined oils (sunflower, soybean), packaged food, cold water, alcohol.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Safe grains to eat: Rice, idli, dosa, jowar roti, bajra roti, ragi roti. Safe dal: Moong dal (best), toor dal.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Safe vegetables and fruits: Drumstick, cucumber with skin, bitter gourd (karela), south Indian greens except spinach, murunga keerai, manathakkali, agathi keerai. Safe fruits: banana, papaya, mango, watermelon (moderate, between meals), muskmelon, amla daily (best — 20x more vitamin C than lemon).",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Safe fats, nuts, and protein: Ghee and coconut oil for cooking. Almonds soaked overnight with skin peeled, walnuts small handful, cashews. Fresh fish and eggs for protein.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Safe drinks: Warm water, warm lemon water in morning, warm water with jaggery in morning. Black tea weak after breakfast with 30-45 min gap. Milk tea only after breakfast, less milk, jaggery, one cup only.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Safe snacks: Roasted makhana, coconut pieces, soaked almonds and walnuts, cucumber, banana.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Natural antihistamines to take daily: Amla is best — 20x vitamin C than lemon. Turmeric with black pepper. Ginger. Lemon in small quantity. Onion and apple contain quercetin which helps.",
        'source': 'health',
        'domain': 'health',
    },
    {
        'content': "Cooking alternatives for psoriasis: Use kokum instead of tomato for sourness. Tamarind if no acidity. Turmeric and black pepper daily. Pepper rasam without tomato.",
        'source': 'health',
        'domain': 'health',
    },
]

print(f"Health chunks prepared: {len(health_chunks)}")


## 5. Embed and Store All Chunks

In [ ]:

def embed_and_store(chunks, conn, cur):
    """Embed all chunks and store in SQLite."""
    # Clear existing embeddings for clean run
    cur.execute("DELETE FROM embeddings")
    
    stored = 0
    for chunk in chunks:
        embedding = model.encode(chunk['content'])
        embedding_blob = embedding.astype(np.float32).tobytes()
        
        cur.execute(
            "INSERT INTO embeddings (domain, content, embedding, source) VALUES (?, ?, ?, ?)",
            (chunk['domain'], chunk['content'], embedding_blob, chunk.get('source'))
        )
        stored += 1
    
    conn.commit()
    print(f"Stored {stored} chunks in vector store.")
    return stored

all_chunks = investment_chunks + health_chunks
total = embed_and_store(all_chunks, conn, cur)
print(f"Total chunks embedded: {total}")


## 6. Cosine Similarity Search

In [ ]:

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search_notes(query, domain, top_k=3):
    """
    Search vector store for most relevant chunks.
    Returns list of (score, content, source) tuples.
    """
    query_embedding = model.encode(query).astype(np.float32)
    
    cur.execute("SELECT content, embedding, source FROM embeddings WHERE domain = ?", (domain,))
    rows = cur.fetchall()
    
    if not rows:
        return []
    
    results = []
    for row in rows:
        stored_embedding = np.frombuffer(row['embedding'], dtype=np.float32)
        score = cosine_similarity(query_embedding, stored_embedding)
        results.append((score, row['content'], row['source']))
    
    results.sort(key=lambda x: x[0], reverse=True)
    return results[:top_k]

print("Search function defined.")


## 7. Test Queries — Investment Domain

In [ ]:

investment_tests = [
    "What did Anand say about Cipla",
    "What did Anand say about South Indian Bank",
    "What did Anand say about IndusInd Bank",
    "What mistakes did I make in investments",
    "What is my pharma allocation plan",
    "What is the 48 hour rule",
    "Peter Lynch investing principles",
    "What stocks do I currently hold",
]

print("Investment search results:")
print("="*60)
for query in investment_tests:
    results = search_notes(query, 'investment', top_k=1)
    if results:
        score, content, source = results[0]
        preview = content[:120] + "..." if len(content) > 120 else content
        print(f"\nQ: {query}")
        print(f"Score: {score:.3f} | Source: {source}")
        print(f"Result: {preview}")
    else:
        print(f"\nQ: {query}")
        print("NO RESULTS FOUND")


## 8. Test Queries — Health Domain

In [ ]:

health_tests = [
    "Can I eat tomato",
    "Can I eat brinjal",
    "Can I drink milk",
    "What can I eat for snacks",
    "What are natural antihistamines",
    "Can I eat prawns",
    "What grains can I eat",
    "What damages the gut",
]

print("Health search results:")
print("="*60)
for query in health_tests:
    results = search_notes(query, 'health', top_k=1)
    if results:
        score, content, source = results[0]
        preview = content[:120] + "..." if len(content) > 120 else content
        print(f"\nQ: {query}")
        print(f"Score: {score:.3f}")
        print(f"Result: {preview}")
    else:
        print(f"\nQ: {query}")
        print("NO RESULTS FOUND")


## 9. Exit Criteria Check

In [ ]:

print("Running exit criteria check...")
all_tests = [
    ("What did Anand say about Cipla",       'investment', 'anand'),
    ("What mistakes did I make in investments", 'investment', 'self'),
    ("What is my pharma allocation plan",    'investment', 'self'),
    ("Can I eat tomato",                     'health',     'health'),
    ("Can I drink milk",                     'health',     'health'),
    ("What are natural antihistamines",      'health',     'health'),
]

passed = 0
for query, domain, expected_source in all_tests:
    results = search_notes(query, domain, top_k=1)
    if results:
        score, content, source = results[0]
        if score > 0.3:  # minimum relevance threshold
            print(f"✓ score={score:.3f} | {query}")
            passed += 1
        else:
            print(f"✗ LOW SCORE {score:.3f} | {query}")
    else:
        print(f"✗ NO RESULT | {query}")

pct = passed / len(all_tests) * 100
print(f"\nScore: {passed}/{len(all_tests)} = {pct:.1f}%")
print("EXIT CRITERIA: >= 80% relevant results")
print("PASS ✓" if pct >= 80 else "FAIL ✗ — check chunking or model")

conn.close()
